# Task 0.4: Neighborhood Clustering

**Spec Reference:** Lines 2845-2847, Appendix A Lines 4453-4478

Creates neighborhood mapping using K-Means clustering.

In [ ]:
# Cell 1: Setup
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import json
from pathlib import Path

In [ ]:
# Cell 2: Load data
df = pd.read_parquet('../data/raw/yellow_tripdata_2024-01.parquet')
print(f"Loaded {len(df):,} records")

In [ ]:
# Cell 3: Aggregate by zone
zone_stats = df.groupby('PULocationID').agg({
    'trip_distance': ['mean', 'std'],
    'fare_amount': ['mean', 'std'],
    'PULocationID': 'count'
}).round(2)
zone_stats.columns = ['avg_dist', 'std_dist', 'avg_fare', 'std_fare', 'trip_count']
zone_stats = zone_stats.reset_index()
print(f"Aggregated {len(zone_stats)} zones")

In [ ]:
# Cell 4: Identify airport zones
airport_zones = {132, 138}  # JFK, LGA
zone_stats['is_airport'] = zone_stats['PULocationID'].isin(airport_zones)
print(f"Airport zones: {airport_zones}")

In [ ]:
# Cell 5: K-Means clustering (non-airport zones)
non_airport = zone_stats[~zone_stats['is_airport']].copy()
features = non_airport[['avg_dist', 'avg_fare']].values
features_norm = (features - features.mean(axis=0)) / features.std(axis=0)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
non_airport['cluster'] = kmeans.fit_predict(features_norm)
print(f"K-Means completed with 5 clusters")

In [ ]:
# Cell 6: Assign unique neighborhood names per cluster
# Use cluster ID directly to ensure distinct neighborhoods
cluster_names = {
    0: "zone_cluster_0",
    1: "zone_cluster_1",
    2: "zone_cluster_2",
    3: "zone_cluster_3",
    4: "zone_cluster_4"
}

non_airport['neighborhood'] = non_airport['cluster'].map(cluster_names)

# Airport zones
airport_stats = zone_stats[zone_stats['is_airport']].copy()
airport_stats['neighborhood'] = 'zone_airports'

# Combine
final_mapping = pd.concat([non_airport, airport_stats])
print(f"Total zones mapped: {len(final_mapping)}")

In [ ]:
# Cell 7: Validate distribution
neighbor_counts = final_mapping.groupby('neighborhood').agg({
    'trip_count': 'sum',
    'PULocationID': 'count'
})
neighbor_counts['trip_pct'] = (neighbor_counts['trip_count'] / 
                                 neighbor_counts['trip_count'].sum() * 100)

print("Neighborhood Distribution:")
print(neighbor_counts)

max_pct = neighbor_counts['trip_pct'].max()
min_trips = neighbor_counts['trip_count'].min()

print(f"\nMax concentration: {max_pct:.1f}%")
print(f"Min trips: {min_trips:,}")

assert max_pct < 50, f"Neighborhood too concentrated: {max_pct:.1f}%"
assert min_trips > 100_000, f"Neighborhood too small: {min_trips:,}"
print("✅ Distribution validated")

In [ ]:
# Cell 8: Export to JSON
neighborhood_map = {}
for _, row in final_mapping.iterrows():
    zone_id = int(row['PULocationID'])
    neighborhood_map[zone_id] = row['neighborhood']

# Fill missing zones with ALL_ZONES
for zone_id in range(1, 266):
    if zone_id not in neighborhood_map:
        neighborhood_map[zone_id] = "ALL_ZONES"

output_path = Path('../src/config/neighborhood_mapping.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump({
        "version": "1.0",
        "num_neighborhoods": len(set(neighborhood_map.values())),
        "clustering_method": "KMeans(n=5) + airport heuristic",
        "mapping": neighborhood_map
    }, f, indent=2)

print(f"✅ Saved to {output_path}")